# Customer Churn / Lead Scoring with Logistic Regression
### Course template: Machine Learning for Business Insights (Module 4)

**What this notebook does**
1. Loads your CSV and checks data quality
2. Builds a leakage-safe preprocessing pipeline
3. Compares a "predict the most common outcome" baseline against Logistic Regression
4. Tunes regularization (and class weighting) with cross-validation
5. Evaluates with business metrics: precision, recall, F1, ROC-AUC, PR-AUC, lift
6. Chooses a decision threshold using capacity and expected value, not the default 0.5
7. Extracts coefficients as odds ratios and probability changes you can explain

**How to use it**
- Edit **only the CONFIG cell** (Cell 3) on your first run
- Run every cell top to bottom (Runtime → Run all in Colab)
- Read the printed notes after each step

**Leave `CSV_PATH = None`** to run on a built-in demo dataset first.

In [ ]:
# ============================================================
# IMPORTS
# All preinstalled in Google Colab. For local Jupyter:
#   pip install pandas numpy scikit-learn matplotlib seaborn
# Requires scikit-learn 1.2 or newer.
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,           # keeps the same share of "yes" cases in every fold
    RepeatedStratifiedKFold,   # repeats it with different shuffles (small data)
    cross_validate,
    cross_val_predict,         # out-of-fold predictions for honest threshold analysis
    GridSearchCV,
    RandomizedSearchCV,
    learning_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix, accuracy_score,
    RocCurveDisplay, PrecisionRecallDisplay,
)
from sklearn.calibration import calibration_curve
from scipy.stats import loguniform

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

RANDOM_STATE = 42
print("Libraries loaded.")

In [ ]:
# ============================================================
# CONFIG: EDIT THIS CELL FOR YOUR DATA
# ============================================================

# --- 1. Where is your data? ---------------------------------
# e.g. "/content/customers.csv" in Colab (upload via the folder icon first).
# Leave as None to use the built-in demo dataset.
CSV_PATH = None

# --- 2. What are you predicting? ----------------------------
# A yes/no column: churned, converted, paid_late, etc.
TARGET_COLUMN = "churned"

# Which value means "YES" (the outcome you care about)?
# Examples: 1, True, "Yes", "churned", "won"
POSITIVE_VALUE = 1

# --- 3. Which columns are inputs? ---------------------------
NUMERIC_COLUMNS = ["tenure_months", "monthly_fee", "support_tickets_60d",
                   "logins_30d", "discount_pct"]
CATEGORICAL_COLUMNS = ["plan_type", "region", "acquisition_channel"]

# IDs and LEAKAGE columns: anything you would NOT know on the scoring date.
# Leakage examples: "cancellation_reason", "contract_end_date", "final_invoice".
DROP_COLUMNS = ["customer_id"]

# --- 4. Main metric for tuning -------------------------------
# "roc_auc"           good general ranking metric
# "average_precision" (PR-AUC) better when "yes" is rare (under ~10%)
# "recall"            when missing a case is very costly
# "f1"                balance of precision and recall
MAIN_METRIC = "average_precision"

# --- 5. Business economics (rough estimates are fine) -------
VALUE_IF_SAVED = 1_200      # value of one successful "yes" you act on (e.g., a retained customer's yearly value)
ACTION_COST = 80            # cost of acting on one case (call, offer, visit)
ACTION_SUCCESS_RATE = 0.30  # share of true "yes" cases the action actually saves/converts
TEAM_CAPACITY = None        # max cases your team can act on per period, e.g. 200 (None = no limit)

# --- 6. Data size and tuning --------------------------------
DATA_SIZE_MODE = "auto"     # "auto", "small", or "large"
TUNING_METHOD = "grid"      # "grid" (small data) or "random" (larger data)
SAMPLE_ROWS_FOR_EXPLORATION = None   # e.g. 200_000 for huge files
TEST_SIZE = 0.2

print("Config set. Target:", TARGET_COLUMN, "| Positive value:", POSITIVE_VALUE)

In [ ]:
# ============================================================
# LOAD DATA
# Demo data is SYNTHETIC. Don't draw real conclusions from it.
# ============================================================

def make_demo_data(n=1_500, seed=RANDOM_STATE):
    """Creates a synthetic customer churn dataset."""
    rng = np.random.default_rng(seed)
    plan = rng.choice(["monthly", "annual"], n, p=[0.65, 0.35])
    region = rng.choice(["North", "South", "West"], n)
    channel = rng.choice(["organic", "paid_ads", "referral", "partner"], n, p=[0.4, 0.3, 0.2, 0.1])
    tenure = rng.gamma(2.0, 10, n).round(0) + 1
    fee = rng.normal(60, 15, n).clip(15, 150).round(2)
    tickets = rng.poisson(1.2, n)
    logins = rng.poisson(12, n)
    discount = rng.choice([0, 5, 10, 20], n, p=[0.6, 0.2, 0.15, 0.05])

    # True churn log-odds (hidden "real world" relationship)
    z = (-1.2
         - 0.04 * tenure
         + 0.015 * fee
         + 0.45 * tickets
         - 0.06 * logins
         - 0.02 * discount
         + np.where(plan == "annual", -1.1, 0)
         + np.where(channel == "referral", -0.5, 0)
         + rng.normal(0, 0.5, n))
    churn = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)

    df = pd.DataFrame({
        "customer_id": [f"C{i:05d}" for i in range(n)],
        "plan_type": plan, "region": region, "acquisition_channel": channel,
        "tenure_months": tenure, "monthly_fee": fee,
        "support_tickets_60d": tickets, "logins_30d": logins.astype(float),
        "discount_pct": discount, "churned": churn,
    })
    df.loc[rng.choice(n, 30, replace=False), "logins_30d"] = np.nan
    df.loc[rng.choice(n, 15, replace=False), "acquisition_channel"] = np.nan
    return df

if CSV_PATH is None:
    df = make_demo_data()
    print("Using DEMO data (synthetic). Set CSV_PATH in the CONFIG cell to use your own.")
else:
    df = pd.read_csv(CSV_PATH)
    print(f"Loaded {CSV_PATH}")

if SAMPLE_ROWS_FOR_EXPLORATION and len(df) > SAMPLE_ROWS_FOR_EXPLORATION:
    # Stratified sample keeps the yes/no balance intact
    df = df.groupby(TARGET_COLUMN, group_keys=False).apply(
        lambda g: g.sample(frac=SAMPLE_ROWS_FOR_EXPLORATION / len(df), random_state=RANDOM_STATE))
    print(f"Sampled ~{len(df):,} rows for exploration. Set back to None for final results.")

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

In [ ]:
# ============================================================
# CONFIG CHECK + BINARY TARGET
# Converts your target to 1 (yes) and 0 (no).
# ============================================================

all_cols = [TARGET_COLUMN] + NUMERIC_COLUMNS + CATEGORICAL_COLUMNS
missing = [c for c in all_cols if c not in df.columns]
if missing:
    raise ValueError(f"Columns in CONFIG but not in data: {missing}\nYour columns: {list(df.columns)}")

# Clean numeric columns stored as text (e.g., "$45.00", "12%")
for col in NUMERIC_COLUMNS:
    if not pd.api.types.is_numeric_dtype(df[col]):
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(r"[$,%\s]", "", regex=True), errors="coerce")
        print(f"Converted '{col}' from text to numbers.")

df = df.dropna(subset=[TARGET_COLUMN])

# Build the 0/1 target. Compare as strings so 1 / "1" / True all work.
df["_y"] = (df[TARGET_COLUMN].astype(str).str.strip().str.lower()
            == str(POSITIVE_VALUE).strip().lower()).astype(int)
if df["_y"].nunique() < 2:
    raise ValueError(f"After mapping, the target has only one class. Check POSITIVE_VALUE. "
                     f"Values found: {df[TARGET_COLUMN].unique()[:10]}")
print("Target mapped: 1 = YES (", POSITIVE_VALUE, "), 0 = NO")

## Section 1: Quick EDA and Data Quality

Check for blanks, impossible values, duplicates, typos in categories, and, most importantly,
**how rare the "yes" outcome is**. That number shapes everything that follows.

In [ ]:
# ============================================================
# DATA QUALITY REPORT
# ============================================================

print("=== Missing values ===")
miss = df[NUMERIC_COLUMNS + CATEGORICAL_COLUMNS].isna().sum()
print(miss[miss > 0] if miss.sum() else "No missing values.", "\n")

print("=== Duplicates ===", df.duplicated().sum(), "exact duplicate rows\n")

print("=== Numeric summary ===")
display(df[NUMERIC_COLUMNS].describe().T)

print("=== Category counts (watch for typos and very rare values) ===")
for col in CATEGORICAL_COLUMNS:
    print(f"\n{col}:"); print(df[col].value_counts(dropna=False))

# ---- Class balance: the most important number in this notebook ----
n_yes, n_no = int(df["_y"].sum()), int((1 - df["_y"]).sum())
yes_rate = n_yes / len(df)
print(f"\n=== CLASS BALANCE ===\nYES: {n_yes:,} ({yes_rate:.1%})   NO: {n_no:,} ({1 - yes_rate:.1%})")
if yes_rate < 0.10:
    print("YES is rare (<10%). Accuracy will be misleading; focus on PR-AUC, recall, and precision.")

In [ ]:
# ============================================================
# QUICK DRIVER VIEW (before any modeling)
# This is the "pivot table first" check from Module 1.
# Numeric: compare averages for YES vs. NO.
# Categorical: YES rate per category.
# ============================================================

print("=== Numeric inputs: average for YES vs. NO ===")
display(df.groupby("_y")[NUMERIC_COLUMNS].mean().T.rename(columns={0: "NO avg", 1: "YES avg"}))

fig, axes = plt.subplots(1, len(CATEGORICAL_COLUMNS), figsize=(5 * len(CATEGORICAL_COLUMNS), 3.5))
axes = np.atleast_1d(axes)
for ax, col in zip(axes, CATEGORICAL_COLUMNS):
    rates = df.groupby(col)["_y"].mean().sort_values()
    rates.plot.barh(ax=ax, color="steelblue")
    ax.axvline(yes_rate, color="red", ls="--", label="overall")
    ax.set_title(f"YES rate by {col}"); ax.set_xlabel("YES rate"); ax.legend()
plt.tight_layout(); plt.show()

# Overlapping numeric inputs make coefficients unstable
corr = df[NUMERIC_COLUMNS].corr()
for i, a in enumerate(NUMERIC_COLUMNS):
    for b in NUMERIC_COLUMNS[i + 1:]:
        if abs(corr.loc[a, b]) > 0.8:
            print(f"High overlap: {a} & {b} (r = {corr.loc[a, b]:.2f}). Consider keeping only one.")

In [ ]:
# ============================================================
# DATA SIZE CHECK
# For yes/no problems, the SMALLER group sets your limits.
# Rule of thumb: about 10–20 rare-group cases per input feature.
# ============================================================

n_minority = min(n_yes, n_no)
n_features_est = len(NUMERIC_COLUMNS) + sum(df[c].nunique() - 1 for c in CATEGORICAL_COLUMNS)
events_per_feature = n_minority / max(n_features_est, 1)

MODE = ("small" if n_minority < 200 else "large") if DATA_SIZE_MODE == "auto" else DATA_SIZE_MODE

print(f"Rare-group cases: {n_minority:,}  |  Estimated features: {n_features_est}")
print(f"Rare-group cases per feature: {events_per_feature:.1f}")
print(f"Mode: {MODE.upper()}")
if events_per_feature < 10:
    print("\nWARNING: fewer than 10 rare-group cases per feature. High overfitting risk.\n"
          "  → Cut inputs to the ones with a clear business reason, group rare categories,\n"
          "    and expect the tuning step to choose strong regularization.")

## Section 2: Train/Test Split and Validation Strategy

- The **test set** (20%) is locked away until the end.
- **Stratified** splitting keeps the YES rate the same in train, test, and every CV fold.
  Without it, a fold might end up with almost no YES cases.
- Small data uses **repeated** stratified CV for steadier scores.

> **If your data is time-based** (e.g., you score customers monthly), the honest approach is to
> train on earlier months and test on later ones. This template uses a random stratified split for
> simplicity; see the Module 7 notes for time-based validation.

In [ ]:
# ============================================================
# SPLIT + CROSS-VALIDATION
# ============================================================

feature_cols = NUMERIC_COLUMNS + CATEGORICAL_COLUMNS
X = df[feature_cols]
y = df["_y"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)

print(f"Train: {len(X_train):,} rows ({y_train.mean():.1%} YES)  |  "
      f"Test: {len(X_test):,} rows ({y_test.mean():.1%} YES)")

if MODE == "small":
    CV = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)
    print("CV: RepeatedStratifiedKFold (5 folds × 5 repeats)")
    if y_train.sum() < 25:
        CV = RepeatedStratifiedKFold(n_splits=3, n_repeats=10, random_state=RANDOM_STATE)
        print("Very few YES cases → switched to 3 folds × 10 repeats so each fold has enough YES cases.")
else:
    CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    print("CV: StratifiedKFold (5 folds)")

## Section 3: Leakage-Safe Preprocessing Pipeline

All prep steps live **inside** the pipeline, so during cross-validation they learn only
from training folds.

| Column type | Steps |
|---|---|
| Numeric | Fill blanks (median) → scale |
| Categorical | Fill blanks ("Unknown") → one-hot encode, dropping one **reference** category |

Scaling is required because regularization penalizes coefficient size; inputs must be comparable.

In [ ]:
# ============================================================
# PREPROCESSOR + MODEL FACTORY
# ============================================================

numeric_steps = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
# REFERENCE GROUPS: the most common value of each category (from TRAINING rows).
# Odds ratios for the other values read as "compared to the most common group".
REFERENCE_GROUPS = [X_train[c].mode().iloc[0] for c in CATEGORICAL_COLUMNS]
print("Reference groups:", {c: str(r) for c, r in zip(CATEGORICAL_COLUMNS, REFERENCE_GROUPS)})

categorical_steps = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
    # drop=REFERENCE_GROUPS: the reference value gets no column of its own.
    # handle_unknown="ignore": new categories in future data won't crash scoring.
    ("onehot", OneHotEncoder(drop=REFERENCE_GROUPS, handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("num", numeric_steps, NUMERIC_COLUMNS),
    ("cat", categorical_steps, CATEGORICAL_COLUMNS),
])

def make_model(clf):
    """Wraps prep + classifier into one leakage-safe pipeline."""
    return Pipeline([("prep", preprocessor), ("model", clf)])

print("Preprocessor ready.")

## Section 4: Baseline and Default Models

- **Baseline:** always predict the most common outcome. For rare YES, this gets high *accuracy* while catching **zero** YES cases, which is exactly why accuracy misleads.
- **Logistic Regression (default):** `C=1.0`, mild regularization (Ridge-style "L2" penalty).
- **Logistic Regression (balanced):** `class_weight="balanced"` tells the model that mistakes on the rare YES group count more. Usually raises recall.

In [ ]:
# ============================================================
# MODEL COMPARISON (cross-validated, TRAINING data only)
# ============================================================

SCORING = {
    "Accuracy": "accuracy",
    "Precision": "precision",
    "Recall": "recall",
    "F1": "f1",
    "ROC-AUC": "roc_auc",
    "PR-AUC": "average_precision",
}

candidates = {
    "Baseline (most common)": make_model(DummyClassifier(strategy="most_frequent")),
    "Logistic (default)": make_model(LogisticRegression(max_iter=5_000)),
    "Logistic (balanced)": make_model(LogisticRegression(max_iter=5_000, class_weight="balanced")),
}

def cv_summary(model, name):
    res = cross_validate(model, X_train, y_train, cv=CV, scoring=SCORING, error_score=np.nan)
    out = {"model": name}
    for m in SCORING:
        out[m] = np.nanmean(res[f"test_{m}"])
        out[f"{m} ±"] = np.nanstd(res[f"test_{m}"])
    return out

results = pd.DataFrame([cv_summary(m, n) for n, m in candidates.items()]).set_index("model")
display(results[["Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "ROC-AUC ±", "PR-AUC", "PR-AUC ±"]])

print(f"""
HOW TO READ THIS (all at the default 0.5 threshold, except the AUCs)
- Accuracy: share of all calls that were right. MISLEADING when YES is rare: compare
  the baseline's accuracy to the others.
- Precision: of the cases we flagged, how many were real YES? (Cost of false alarms.)
- Recall: of all real YES cases, how many did we catch? (Cost of misses.)
- F1: balance of precision and recall.
- ROC-AUC: how well the model RANKS YES above NO, at any threshold. 0.5 = coin flip, 1.0 = perfect.
- PR-AUC: like ROC-AUC but stricter when YES is rare. Compare it to the YES rate
  ({y_train.mean():.2f}), which is what random guessing would score.
- ±: spread across folds. Differences smaller than the spread = effectively tied.
""")

## Section 5: Hyperparameter Tuning

Logistic regression's key setting is **C**, the inverse of regularization strength:
- **Small C** (e.g., 0.01) → very cautious, coefficients shrink strongly toward "no effect"
- **Large C** (e.g., 100) → nearly unregularized

We also test **class weight**: `None` vs. `"balanced"` (extra weight on the rare YES group).

We keep the default Ridge-style (**L2**) penalty, which keeps every driver and is the most stable
choice for business data. (Lasso-style L1, which can drop weak drivers to zero, is covered in Module 7.)

**Simplicity rule:** keeping the best setting's class weight, we pick the **smallest C** (the most
cautious model) whose score is within **one standard error** of the best (the fold spread ÷ √folds).
Equal scores, safer model.

**Small-data notes**
- Keep the grid small and lean toward **smaller C** when scores are close
- Distrust small gains; compare them to the spread

**Large-data notes**
- Wider C ranges and `TUNING_METHOD = "random"` are fine and fast

In [ ]:
# ============================================================
# TUNING
# We score every setting on several metrics, but pick the winner
# by MAIN_METRIC (set in CONFIG). return_train_score=True lets us
# check for overfitting (train score much higher than validation).
# ============================================================

base_clf = LogisticRegression(max_iter=10_000)     # default L2 penalty, lbfgs solver

if MODE == "small":
    C_values = [0.01, 0.03, 0.1, 0.3, 1, 3]         # leans cautious
else:
    C_values = list(np.logspace(-2, 2, 9))           # 0.01 → 100

param_grid = {"model__C": C_values, "model__class_weight": [None, "balanced"]}
model_to_tune = make_model(base_clf)

if TUNING_METHOD == "random":
    dist = {"model__C": loguniform(min(C_values), max(C_values)),
            "model__class_weight": [None, "balanced"]}
    search = RandomizedSearchCV(model_to_tune, dist, n_iter=25, cv=CV, scoring=SCORING,
                                refit=False, random_state=RANDOM_STATE, n_jobs=-1,
                                return_train_score=True)
else:
    search = GridSearchCV(model_to_tune, param_grid, cv=CV, scoring=SCORING,
                          refit=False, n_jobs=-1, return_train_score=True)
search.fit(X_train, y_train)

# Translate MAIN_METRIC (sklearn name) into our display name, e.g. "average_precision" → "PR-AUC"
MAIN_KEY = {v: k for k, v in SCORING.items()}[MAIN_METRIC]
cv = search.cv_results_
tbl = pd.DataFrame({
    "C": [p["model__C"] for p in cv["params"]],
    "class_weight": [str(p["model__class_weight"]) for p in cv["params"]],
    MAIN_KEY: cv[f"mean_test_{MAIN_KEY}"],
    "±": cv[f"std_test_{MAIN_KEY}"],
    f"{MAIN_KEY} (train)": cv[f"mean_train_{MAIN_KEY}"],
    "Recall": cv["mean_test_Recall"],
    "Precision": cv["mean_test_Precision"],
})
tbl = tbl.sort_values(MAIN_KEY, ascending=False).reset_index(drop=True)
print(f"Top 10 settings by {MAIN_KEY}:")
display(tbl.head(10))

# ============================================================
# SIMPLICITY RULE (the "one standard error" rule):
# 1. Keep the class_weight of the best setting.
# 2. Among its C values scoring within ONE STANDARD ERROR of the best
#    (standard error = fold spread ÷ √number of folds), take the
#    SMALLEST C (strongest regularization = most cautious model).
# ============================================================
n_folds = CV.get_n_splits()
best = tbl.iloc[0]
best_se = best["±"] / np.sqrt(n_folds)
within = tbl[(tbl["class_weight"] == best["class_weight"]) & (tbl[MAIN_KEY] >= best[MAIN_KEY] - best_se)]
chosen = within.sort_values(["C", MAIN_KEY], ascending=[True, False]).iloc[0]
BEST_C = float(chosen["C"])
BEST_CW = None if chosen["class_weight"] == "None" else chosen["class_weight"]
print(f"\nBest score: {best[MAIN_KEY]:.3f} ± {best['±']:.3f}  (C={best['C']:.3g}, class_weight={best['class_weight']})")
print(f"CHOSEN (most cautious within one standard error, ±{best_se:.3f}): C={BEST_C:.3g}, class_weight={BEST_CW}  → {chosen[MAIN_KEY]:.3f}")
print("""
NOTE ON class_weight: "balanced" mostly shifts probabilities UP for the YES group, which raises recall
at the 0.5 cutoff. It barely changes ranking metrics (ROC-AUC, PR-AUC). Since we choose our own
threshold in Section 7, either choice is fine. Unbalanced probabilities are easier to read as real risk.""")

final_model = make_model(LogisticRegression(max_iter=10_000, C=BEST_C, class_weight=BEST_CW))

In [ ]:
# ============================================================
# TUNED MODEL vs. BASELINES (same CV folds)
# ============================================================
results2 = pd.concat([results, pd.DataFrame([cv_summary(final_model, "Logistic (tuned)")]).set_index("model")])
display(results2[["Precision", "Recall", "F1", "ROC-AUC", "ROC-AUC ±", "PR-AUC", "PR-AUC ±"]])
gain = results2.loc["Logistic (tuned)", MAIN_KEY] - results2.loc["Logistic (default)", MAIN_KEY]
print(f"Tuning changed {MAIN_KEY} by {gain:+.3f}. "
      + ("That's within the noise; the default was already fine." if abs(gain) < results2.loc['Logistic (tuned)', f'{MAIN_KEY} ±']
         else "That's a real improvement."))

## Section 6: Final Test-Set Evaluation

Fit on all training rows, then score the locked **test set once**. These are the numbers to report.

In [ ]:
# ============================================================
# TEST SET
# ============================================================
final_model.fit(X_train, y_train)
proba_test = final_model.predict_proba(X_test)[:, 1]

print(f"""=== TEST SET: ranking quality ===
ROC-AUC: {roc_auc_score(y_test, proba_test):.3f}   (0.5 = coin flip, 1.0 = perfect)
PR-AUC:  {average_precision_score(y_test, proba_test):.3f}   (random guessing = {y_test.mean():.3f})
""")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
RocCurveDisplay.from_predictions(y_test, proba_test, ax=ax[0], name="Logistic")
ax[0].plot([0, 1], [0, 1], "k--", lw=0.8); ax[0].set_title("ROC curve")
PrecisionRecallDisplay.from_predictions(y_test, proba_test, ax=ax[1], name="Logistic")
ax[1].axhline(y_test.mean(), color="k", ls="--", lw=0.8, label="random"); ax[1].legend()
ax[1].set_title("Precision–recall curve")
plt.tight_layout(); plt.show()

# Confusion matrix at the default 0.5 cutoff, for reference only
pred05 = (proba_test >= 0.5).astype(int)
cm = confusion_matrix(y_test, pred05)
print("At the default 0.5 cutoff (we'll choose a better one next):")
print(pd.DataFrame(cm, index=["Actual NO", "Actual YES"], columns=["Predicted NO", "Predicted YES"]))
print(f"Precision {precision_score(y_test, pred05, zero_division=0):.2f} | "
      f"Recall {recall_score(y_test, pred05):.2f} | F1 {f1_score(y_test, pred05):.2f}")

## Section 7: Choosing the Decision Threshold (Business Value, Not 0.5)

The model gives each customer a **probability**. The **threshold** decides who gets action.
- Lower threshold → more customers flagged → catch more YES cases, but more wasted actions
- Higher threshold → fewer, more confident flags

For each possible threshold we estimate:

**Expected value = (YES cases caught × VALUE_IF_SAVED × ACTION_SUCCESS_RATE) − (customers contacted × ACTION_COST)**

We use **out-of-fold predictions on the training data** (each row scored by a model that never saw it),
so the threshold is chosen without touching the test set. Then we check it once on the test set.

In [ ]:
# ============================================================
# OUT-OF-FOLD PROBABILITIES (honest predictions for every training row)
# cross_val_predict needs each row predicted exactly once, so we use
# plain (non-repeated) stratified folds here.
# ============================================================
OOF_CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
proba_oof = cross_val_predict(final_model, X_train, y_train, cv=OOF_CV, method="predict_proba")[:, 1]

BREAK_EVEN = ACTION_COST / (VALUE_IF_SAVED * ACTION_SUCCESS_RATE)
CUSTOMER_BASE = len(df)   # customers you score each cycle (assumed = rows in your file)

def threshold_table(y_true, proba, value=VALUE_IF_SAVED, cost=ACTION_COST, success=ACTION_SUCCESS_RATE):
    """Expected value of acting on everyone above each threshold, scaled to the full customer base."""
    y_true = np.asarray(y_true)
    cands = list(np.round(np.arange(0.05, 0.951, 0.025), 3))
    if TEAM_CAPACITY:
        # Add the threshold that flags exactly TEAM_CAPACITY customers
        share = min(TEAM_CAPACITY / CUSTOMER_BASE, 1.0)
        cands.append(float(np.quantile(proba, 1 - share)))
    rows = []
    for t in sorted(set(cands)):
        flag = proba >= t
        n_flag, tp = int(flag.sum()), int((flag & (y_true == 1)).sum())
        scale = CUSTOMER_BASE / len(y_true)                   # from this sample to your full base
        ev = (tp * value * success - n_flag * cost) * scale
        rows.append({"threshold": t,
                     "contacted": round(n_flag * scale),
                     "YES caught": round(tp * scale),
                     "precision": tp / n_flag if n_flag else np.nan,
                     "recall": tp / max(y_true.sum(), 1),
                     "expected value": ev,
                     "within capacity": (TEAM_CAPACITY is None) or (n_flag * scale <= TEAM_CAPACITY * 1.001)})
    return pd.DataFrame(rows)

tt = threshold_table(y_train, proba_oof)
ok = tt[tt["within capacity"]]
best_row = ok.loc[ok["expected value"].idxmax()]
THRESHOLD = float(best_row["threshold"])

display(tt.iloc[::2].style.format({"threshold": "{:.3f}", "precision": "{:.2f}", "recall": "{:.2f}",
                                    "expected value": "${:,.0f}"}).hide(axis="index"))
print(f"""
BREAK-EVEN PRECISION: {BREAK_EVEN:.1%}. Acting on a customer pays off only if at least this share
of contacted customers are real YES cases (= ACTION_COST / (VALUE_IF_SAVED × ACTION_SUCCESS_RATE)).

CHOSEN THRESHOLD: {THRESHOLD:.3f}{'  (limited by TEAM_CAPACITY)' if TEAM_CAPACITY else ''}
→ contact about {best_row['contacted']:,.0f} of {CUSTOMER_BASE:,} customers,
  catch about {best_row['recall']:.0%} of YES cases, at {best_row['precision']:.0%} precision.
→ Estimated value per cycle: ${best_row['expected value']:,.0f} (vs. $0 for doing nothing).
""")

plt.figure(figsize=(8, 4))
plt.plot(tt["threshold"], tt["expected value"], "o-")
plt.axvline(THRESHOLD, color="red", ls="--", label=f"chosen {THRESHOLD:.2f}")
plt.axhline(0, color="black", lw=0.8)
plt.xlabel("Threshold"); plt.ylabel("Expected value per cycle ($)"); plt.title("Value of acting at each threshold")
plt.legend(); plt.show()

# ---- One honest check on the test set ----
tt_test = threshold_table(y_test, proba_test)
row_test = tt_test.iloc[(tt_test["threshold"] - THRESHOLD).abs().argmin()]
print(f"TEST SET at threshold {THRESHOLD:.3f}: precision {row_test['precision']:.0%}, "
      f"recall {row_test['recall']:.0%}, estimated value ${row_test['expected value']:,.0f} per cycle.")

In [ ]:
# ============================================================
# SENSITIVITY: what if our business numbers are off?
# Your VALUE_IF_SAVED and ACTION_SUCCESS_RATE are estimates. If the
# best threshold barely moves when they change, the decision is robust.
# ============================================================
sens = []
for v_mult in [0.5, 1.0, 1.5]:
    for s_mult in [0.5, 1.0, 1.5]:
        t_ = threshold_table(y_train, proba_oof, value=VALUE_IF_SAVED * v_mult,
                             success=min(ACTION_SUCCESS_RATE * s_mult, 1.0))
        t_ = t_[t_["within capacity"]]
        r = t_.loc[t_["expected value"].idxmax()]
        sens.append({"value if saved": VALUE_IF_SAVED * v_mult,
                     "success rate": min(ACTION_SUCCESS_RATE * s_mult, 1.0),
                     "best threshold": r["threshold"], "contacted": r["contacted"],
                     "expected value": r["expected value"]})
display(pd.DataFrame(sens).style.format({"value if saved": "${:,.0f}", "success rate": "{:.0%}",
                                          "best threshold": "{:.3f}", "contacted": "{:,.0f}",
                                          "expected value": "${:,.0f}"}).hide(axis="index"))
print("If the expected value stays positive in most rows, the program is worth piloting even with rough estimates.")

# ============================================================
# LIFT TABLE (test set): how concentrated are YES cases at the top?
# Decile 1 = the 10% of customers with the highest scores.
# ============================================================
lift = pd.DataFrame({"y": y_test.values, "p": proba_test})
lift["decile"] = pd.qcut(lift["p"].rank(method="first", ascending=False), 10, labels=range(1, 11))
lt = lift.groupby("decile", observed=True).agg(customers=("y", "size"), yes=("y", "sum"), avg_score=("p", "mean"))
lt["YES rate"] = lt["yes"] / lt["customers"]
lt["lift"] = lt["YES rate"] / lift["y"].mean()
lt["cumulative % of YES caught"] = lt["yes"].cumsum() / lt["yes"].sum() * 100
display(lt.style.format({"avg_score": "{:.2f}", "YES rate": "{:.1%}", "lift": "{:.1f}x",
                         "cumulative % of YES caught": "{:.0f}%"}))
print(f"Top 10% of scores: YES rate is {lt['lift'].iloc[0]:.1f}x the average. "
      f"Top 30% contains {lt['cumulative % of YES caught'].iloc[2]:.0f}% of all YES cases.")

In [ ]:
# ============================================================
# CALIBRATION: when the model says 30%, is it right about 30% of the time?
# Matters if you use the probabilities directly (e.g. expected-loss math).
# class_weight="balanced" pushes probabilities up; that's expected.
# ============================================================
frac_pos, mean_pred = calibration_curve(y_test, proba_test, n_bins=8, strategy="quantile")
plt.figure(figsize=(5, 5))
plt.plot(mean_pred, frac_pos, "o-", label="model")
plt.plot([0, 1], [0, 1], "k--", label="perfect")
plt.xlabel("Predicted probability"); plt.ylabel("Actual YES rate"); plt.title("Calibration (test set)")
plt.legend(); plt.show()

## Section 8: What Drives the Outcome? (Odds Multipliers in Real Units)

A logistic coefficient is hard to read directly, so we convert it to an **odds multiplier**:

- **1.50** → the odds of YES are 50% higher for each one-unit increase
- **0.80** → the odds are 20% lower
- **1.00** → no effect

We show each numeric driver **per 1 real unit** (e.g. one more support ticket) and **per typical change**
(one standard deviation). Categories are compared with the **reference group** (the most common value).

Odds aren't probabilities, so we also run a **what-if**: nudge each driver by a typical amount for every
test customer and see how the average predicted probability moves, in percentage points.

⚠️ These are associations in historical data. They're strong leads for action, not proof of cause.

In [ ]:
# ============================================================
# ODDS MULTIPLIERS
# ============================================================
prep = final_model.named_steps["prep"]
coefs = final_model.named_steps["model"].coef_[0]
names = [n.split("__", 1)[1] for n in prep.get_feature_names_out()]
scaler = prep.named_transformers_["num"].named_steps["scale"]
n_num = len(NUMERIC_COLUMNS)

rows = []
for i, (name, c) in enumerate(zip(names, coefs)):
    if i < n_num:
        sd = scaler.scale_[i]
        rows.append({"driver": name, "type": "numeric", "typical change (1 SD)": sd,
                     "odds × per 1 unit": np.exp(c / sd), "odds × per typical change": np.exp(c)})
    else:
        rows.append({"driver": name, "type": "category vs. reference", "typical change (1 SD)": np.nan,
                     "odds × per 1 unit": np.exp(c), "odds × per typical change": np.exp(c)})
odds = pd.DataFrame(rows)
odds["strength"] = np.abs(np.log(odds["odds × per typical change"]))
odds = odds.sort_values("strength", ascending=False).drop(columns="strength").reset_index(drop=True)
display(odds.style.format({"typical change (1 SD)": "{:,.2f}", "odds × per 1 unit": "{:.2f}",
                           "odds × per typical change": "{:.2f}"}, na_rep="").hide(axis="index"))

ohe = prep.named_transformers_["cat"].named_steps["onehot"]
for col, cats, d in zip(CATEGORICAL_COLUMNS, ohe.categories_, ohe.drop_idx_):
    print(f"Reference group for {col}: '{cats[d]}'")

# ============================================================
# WHAT-IF: average change in predicted probability (percentage points)
# ============================================================
base_p = final_model.predict_proba(X_test)[:, 1].mean()
whatif = []
for i, col in enumerate(NUMERIC_COLUMNS):
    Xw = X_test.copy()
    Xw[col] = Xw[col] + scaler.scale_[i]                 # everyone +1 typical change
    whatif.append({"driver": col, "change applied": f"+{scaler.scale_[i]:,.1f}",
                   "avg probability change (pts)": (final_model.predict_proba(Xw)[:, 1].mean() - base_p) * 100})
whatif = pd.DataFrame(whatif).sort_values("avg probability change (pts)", key=abs, ascending=False)
display(whatif.style.format({"avg probability change (pts)": "{:+.1f}"}).hide(axis="index"))

print("PLAIN-ENGLISH SUMMARY:")
for _, r in odds.head(6).iterrows():
    m = r["odds × per 1 unit"]
    pct = (m - 1) * 100
    word = "higher" if pct > 0 else "lower"
    if r["type"] == "numeric":
        print(f"- Each +1 in {r['driver']}: odds of YES {abs(pct):.0f}% {word} (× {m:.2f}); "
              f"a typical change of {r['typical change (1 SD)']:,.1f} → odds × {r['odds × per typical change']:.2f}.")
    else:
        print(f"- '{r['driver']}': odds of YES {abs(pct):.0f}% {word} than the reference group (× {m:.2f}).")

## Section 9: Small Data vs. Large Data

**Small data (under ~200 YES cases):**
- Expect the tuning step to choose a small C (strong regularization). That's a feature, not a bug
- Keep 5–8 inputs with a clear business story; group rare categories into "Other"
- Repeated stratified CV is used automatically; read the ± column before celebrating any gain
- Report odds multipliers as rough ranges and confirm the top drivers with a pivot table

**Large data (thousands of YES cases):**
- Regularization matters less; default and tuned models often tie
- `TUNING_METHOD = "random"` is fine; for 1M+ rows use `solver="saga"` and explore on a sample first
- Tiny effects can look "significant" but may not be worth acting on
- If the model plateaus, try Random Forest or Gradient Boosting (Module 5) and compare PR-AUC

In [ ]:
# ============================================================
# SAVE THE MODEL + OPTIONAL CALL LIST
# ============================================================
import joblib

# 1) Call list: score customers and rank by risk.
#    DEMO NOTE: here we score every row in the file. In real use, load your
#    CURRENT active customers (same columns) and score them instead.
ID_COLUMN = next((c for c in DROP_COLUMNS if c in df.columns), None)
scored = df[[ID_COLUMN] if ID_COLUMN else []].copy()
scored["probability"] = final_model.predict_proba(df[NUMERIC_COLUMNS + CATEGORICAL_COLUMNS])[:, 1]
scored["expected value of action"] = scored["probability"] * VALUE_IF_SAVED * ACTION_SUCCESS_RATE - ACTION_COST
scored["flagged"] = scored["probability"] >= THRESHOLD
call_list = scored[scored["flagged"]].sort_values("probability", ascending=False)
if TEAM_CAPACITY:
    call_list = call_list.head(TEAM_CAPACITY)
call_list.round(3).to_csv("logistic_call_list.csv", index=False)
print(f"Saved: logistic_call_list.csv ({len(call_list):,} customers to contact)")
display(call_list.head(10))

# 2) Model: refit on train + test for use going forward
final_model.fit(pd.concat([X_train, X_test]), pd.concat([y_train, y_test]))
joblib.dump({"model": final_model, "threshold": THRESHOLD}, "logistic_model.joblib")
print(f"Saved: logistic_model.joblib (pipeline + threshold {THRESHOLD:.3f})")
# Later: bundle = joblib.load("logistic_model.joblib")
#        p = bundle["model"].predict_proba(new_df[NUMERIC_COLUMNS + CATEGORICAL_COLUMNS])[:, 1]
#        to_contact = new_df[p >= bundle["threshold"]]

## Final Checklist

- [ ] Target mapped correctly (check the YES rate matches what you expect)
- [ ] No leakage: every input is known **before** the scoring date
- [ ] The model clearly beats the "most common outcome" baseline on PR-AUC / ROC-AUC
- [ ] C and class weight tuned with cross-validation; simplest setting within one spread chosen
- [ ] Threshold chosen from **out-of-fold** predictions and business value, not 0.5
- [ ] Test set used **once**; results reported with precision, recall, and expected value
- [ ] Sensitivity table shows the program pays off even if my estimates are off
- [ ] Top drivers explained as odds multipliers in real units, and they make business sense
- [ ] Call list sized to team capacity; plan to measure results against a control group
- [ ] Model and threshold saved, with the training date noted